# Drought duration

This how-to guide demonstrates how to download and process **drought duration** for a specific administratice (NUTS2) region. In this how-to, we retrieve both historical reanalysis and climate projection data for the region EL64 (Central Greece) and save it in a csv file. 

```{admonition} Task
:class: tip
Create a csv-file that contains climate projections of drought duration for a specific NUTS2 region.
```

```{admonition} Outputs
:class: note
This notebook will create the following CSV files in `data/{admin_id}/drought_hazard/`:
- `drought_duration_historical_{admin_id}.csv` - Historical drought duration from reanalysis
- `drought_duration_projections_{admin_id}.csv` - All climate model projections of drought duration
- `drought_duration_all_{admin_id}.csv` - Combined historical and projections
```

**Dataset:** [European Climate Data Explorer - Climate Indicators](https://cds.climate.copernicus.eu/datasets/sis-ecde-climate-indicators)

## Settings

### User settings

In [21]:
admin_id = "EL64"
workdir = "/home/nejk/code/drought_exposure"

### Setup of environment

In [22]:
import os
import zipfile
import cdsapi
import glob
from pathlib import Path
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from functools import partial
from tqdm.auto import tqdm

# Set working directory (adjust as needed)
os.chdir(workdir)
print(f"Working directory: {os.getcwd()}")

Working directory: /etc/ecmwf/nfs/dh2_home_a/nejk/code/drought_exposure


In [23]:
# Define data directories
data_dir = Path("./data")
ecde_dir = data_dir / "ecde"
hist_dir = ecde_dir / "historical"
proj_dir = ecde_dir / "projections"
output_dir = data_dir / "EL64" / "drought_hazard"

# Create directories if they don't exist
for directory in [ecde_dir, hist_dir, proj_dir, output_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Data will be saved to: {ecde_dir}")
print(f"Output files will be saved to: {output_dir}")

Data will be saved to: data/ecde
Output files will be saved to: data/EL64/drought_hazard


### Setup region specifics

In [24]:
regions_dir = data_dir / 'regions'
nuts_shp = regions_dir / 'NUTS_RG_20M_2024_4326' / 'NUTS_RG_20M_2024_4326.shp'
nuts_gdf = gpd.read_file(nuts_shp)

# Select the region of interest
sel_gdf = nuts_gdf[nuts_gdf['NUTS_ID'] == admin_id]
print(f"Region: {sel_gdf['NUTS_NAME'].values[0]}")
print(f"NUTS ID: {admin_id}")
print(f"Country: {sel_gdf['CNTR_CODE'].values[0]}")
print(f"Bounding box: {sel_gdf.geometry.total_bounds}")

Region: Στερεά Ελλάδα
NUTS ID: EL64
Country: EL
Bounding box: [21.39637798 37.98898161 24.67199242 39.27219519]


## Download Hazard Data

### Helper function to unzip downloaded data

The CDS API downloads data as zip files. We need a function to extract and rename the NetCDF files.

In [25]:
def unzip_ecde_data(zipfile_path):
    """
    Unzip ECDE data downloaded from CDS and rename to .nc extension.
    
    Parameters:
    -----------
    zipfile_path : Path or str
        Path to the zip file to extract
    """
    zipfile_path = Path(zipfile_path)
    
    with zipfile.ZipFile(zipfile_path, 'r') as zip_ref:
        names = zip_ref.namelist()
        zip_ref.extractall(zipfile_path.parent)
        
        for name in names:
            if name.split(".")[-1] == "nc":
                # Rename the extracted file to match the zip filename
                nc_path = zipfile_path.with_suffix('.nc')
                os.rename(zipfile_path.parent / name, nc_path)
                print(f"  Extracted: {nc_path.name}")
            else:
                # Remove non-NetCDF files
                os.remove(zipfile_path.parent / name)
    
    # Remove the zip file after extraction
    os.remove(zipfile_path)
    print(f"  Removed: {zipfile_path.name}")

### Download historical drought duration data

We'll download historical drought duration data from the ECDE reanalysis dataset. To avoid long queues in the CDS, we download the data in smaller chunks (e.g., decade by decade).

In [26]:
# Define time periods to download in chunks to avoid long queues
time_chunks = [
    (1979, 1990),
    (1991, 2000),
    (2001, 2010),
    (2011, 2023)
]

# Download historical data
print("Downloading historical drought duration data...")
print("=" * 60)

hist_files = []

for start_year, end_year in time_chunks:
    years = [str(y) for y in range(start_year, end_year + 1)]
    hist_zipfile = hist_dir / f"drought_duration_nuts2_hist_{start_year}_{end_year}.zip"
    hist_ncfile = hist_zipfile.with_suffix('.nc')
    
    # Skip if already downloaded
    if hist_ncfile.exists():
        print(f"Years {start_year}-{end_year}: Already exists, skipping download")
        hist_files.append(hist_ncfile)
        continue
    
    print(f"Years {start_year}-{end_year}: Downloading...")
    
    dataset = "sis-ecde-climate-indicators"
    request = {
        "variable": ["duration_of_meteorological_droughts"],
        "origin": "reanalysis",
        "temporal_aggregation": ["yearly"],
        "spatial_aggregation": "regional_layer",
        "regional_layer": ["nuts_level_2"],
        "year": years
    }
    
    try:
        client = cdsapi.Client()
        client.retrieve(dataset, request).download(str(hist_zipfile))
        unzip_ecde_data(hist_zipfile)
        hist_files.append(hist_ncfile)
        print(f"  ✓ Download complete for {start_year}-{end_year}")
    except Exception as e:
        print(f"  ✗ Error downloading {start_year}-{end_year}: {e}")

print("=" * 60)
print(f"Historical data download complete! Downloaded {len(hist_files)} files.")

Years 1979-1990: Already exists, skipping download
Years 1991-2000: Already exists, skipping download
Years 2001-2010: Already exists, skipping download
Years 2011-2023: Already exists, skipping download
Historical data download complete! Downloaded 4 files.


### Download drought projections

We'll download climate projection data for multiple Global Climate Models (GCMs), Regional Climate Models (RCMs), scenarios (RCPs), and ensemble members. To avoid overwhelming the CDS queue, we download one combination at a time.

We'll download a subset of available model combinations. You can adjust these lists based on your needs.

In [27]:
# Define model combinations
# Note: Not all combinations are available in the dataset

gcms = ["ec_earth", "hadgem2_es", "ipsl_cm5a_mr", "mpi_esm_lr", "noresm1_m"]
rcms = ["cclm4_8_17", "hirham5", "racmo22e", "rca4", "wrf381p"]
rcps = ["rcp4_5", "rcp8_5"]
enss = ["r12i1p1", "r1i1p1", "r3i1p1"]

# Time periods for projections (download in chunks)
proj_time_chunks = [
    (2006, 2025),
    (2026, 2050),
    (2051, 2075),
    (2076, 2100)
]

print(f"Model combinations to attempt:")
print(f"  GCMs: {len(gcms)}")
print(f"  RCMs: {len(rcms)}")
print(f"  RCPs: {len(rcps)}")
print(f"  Ensemble members: {len(enss)}")
print(f"  Time periods: {len(proj_time_chunks)}")
print(f"  Maximum possible downloads: {len(gcms) * len(rcms) * len(rcps) * len(enss) * len(proj_time_chunks)}")
print(f"\nNote: Not all combinations are available in the dataset; we will check for their availability.")

Model combinations to attempt:
  GCMs: 5
  RCMs: 5
  RCPs: 2
  Ensemble members: 3
  Time periods: 4
  Maximum possible downloads: 600

Note: Not all combinations are available in the dataset; we will check for their availability.


In [ ]:
# Download projection data
print("Downloading drought projection data...")
print("=" * 60)

proj_files = []
download_count = 0
skip_count = 0
error_count = 0

# Calculate total iterations for progress bar
total_iterations = len(gcms) * len(rcms) * len(rcps) * len(enss) * len(proj_time_chunks)

# Create progress bar
with tqdm(total=total_iterations, desc="Processing model combinations") as pbar:
    for gcm in gcms:
        for rcm in rcms:
            for rcp in rcps:
                for ens in enss:
                    for start_year, end_year in proj_time_chunks:
                        
                        years = [str(y) for y in range(start_year, end_year + 1)]
                        
                        # Create filename
                        proj_zipfile = proj_dir / f"drought_duration_nuts2_{gcm}_{rcm}_{rcp}_{ens}_{start_year}_{end_year}.zip"
                        proj_ncfile = proj_zipfile.with_suffix('.nc')
                        
                        # Skip if already downloaded
                        if proj_ncfile.exists():
                            skip_count += 1
                            proj_files.append(proj_ncfile)
                            pbar.update(1)
                            continue
                        
                        model_info = f"{gcm}/{rcm}/{rcp}/{ens} ({start_year}-{end_year})"
                        pbar.set_postfix_str(f"Downloading: {model_info}")
                        
                        dataset = "sis-ecde-climate-indicators"
                        request = {
                            "variable": ["duration_of_meteorological_droughts"],
                            "origin": "projections",
                            "gcm": gcm,
                            "rcm": rcm,
                            "experiment": rcp,
                            "ensemble_member": ens,
                            "temporal_aggregation": ["yearly"],
                            "spatial_aggregation": "regional_layer",
                            "regional_layer": ["nuts_level_2"],
                            "year": years
                        }
                        
                        try:
                            client = cdsapi.Client()
                            client.retrieve(dataset, request).download(str(proj_zipfile))
                            unzip_ecde_data(proj_zipfile)
                            proj_files.append(proj_ncfile)
                            download_count += 1
                        except Exception as e:
                            error_count += 1
                            # Silently count errors without printing each one
                        
                        pbar.update(1)

print("=" * 60)
print(f"Projection data download summary:")
print(f"  New downloads: {download_count}")
print(f"  Already existed: {skip_count}")
print(f"  Errors/unavailable: {error_count}")
print(f"  Total projection files: {len(proj_files)}")

Processing model combinations:   0%|          | 0/600 [00:00<?, ?it/s]

Recovering from HTTP error [429 Too Many Requests], attempt 1 of 500
Retrying in 120 seconds


## Process Hazard Data

### Process historical data

Now we'll read the historical drought duration data and extract information for our region (EL64).

In [ ]:
# Read all historical NetCDF files
hist_nc_files = sorted(hist_dir.glob("drought_duration_nuts2_hist_*.nc"))
print(f"Found {len(hist_nc_files)} historical data files")

# Combine all historical files
if len(hist_nc_files) > 0:
    # Use combine='nested' with concat_dim to properly concatenate along time dimension
    drought_dur_hist = xr.open_mfdataset(hist_nc_files, combine='nested', concat_dim='time')
    print("\nHistorical dataset:")
    print(drought_dur_hist)
    
    # Extract data for EL64
    drought_dur_hist_el64 = drought_dur_hist.sel(nuts=admin_id)
    print(f"\nData for {admin_id}:")
    print(drought_dur_hist_el64)
else:
    print("No historical data files found. Please download the data first.")

Found 4 historical data files


/usr/local/apps/python3/3.12.11-01/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.42.0 or higher is recommended. You are running version 2.31.0
  warnings.warn(



Historical dataset:
<xarray.Dataset> Size: 906kB
Dimensions:      (nuts: 334, time: 336)
Coordinates:
  * nuts         (nuts) <U4 5kB 'DE50' 'DE60' 'DE71' ... 'NO06' 'NO07' 'NO08'
  * time         (time) datetime64[ns] 3kB 1940-01-01 1941-01-01 ... 2023-01-01
    realization  int64 8B 0
Data variables:
    dmd          (nuts, time) float64 898kB dask.array<chunksize=(334, 84), meta=np.ndarray>

Data for EL64:
<xarray.Dataset> Size: 5kB
Dimensions:      (time: 336)
Coordinates:
  * time         (time) datetime64[ns] 3kB 1940-01-01 1941-01-01 ... 2023-01-01
    realization  int64 8B 0
    nuts         <U4 16B 'EL64'
Data variables:
    dmd          (time) float64 3kB dask.array<chunksize=(84,), meta=np.ndarray>


### Process projection data

We'll read all projection files and organize them by model combination.

In [ ]:
# Find all projection NetCDF files
proj_nc_files = sorted(proj_dir.glob("drought_duration_nuts2_*.nc"))
print(f"Found {len(proj_nc_files)} projection data files")

# Group files by model combination (gcm_rcm_rcp_ens)
from collections import defaultdict

model_groups = defaultdict(list)

for file in proj_nc_files:
    # Extract model info from filename
    # Format: drought_duration_nuts2_{gcm}_{rcm}_{rcp}_{ens}_{start}_{end}.nc
    parts = file.stem.split('_')
    if len(parts) >= 8:
        gcm, rcm, rcp, ens = parts[4], parts[5], parts[6], parts[7]
        model_key = f"{gcm}_{rcm}_{rcp}_{ens}"
        model_groups[model_key].append(file)

print(f"\nFound {len(model_groups)} unique model combinations:")
for i, (model_key, files) in enumerate(list(model_groups.items())[:5]):
    print(f"  {i+1}. {model_key}: {len(files)} time chunks")
if len(model_groups) > 5:
    print(f"  ... and {len(model_groups) - 5} more")

Found 72 projection data files

Found 17 unique model combinations:
  1. earth_hirham5_rcp4_5: 4 time chunks
  2. earth_hirham5_rcp8_5: 4 time chunks
  3. earth_racmo22e_rcp4_5: 4 time chunks
  4. earth_racmo22e_rcp8_5: 4 time chunks
  5. earth_rca4_rcp4_5: 4 time chunks
  ... and 12 more


In [ ]:
# Load projection data for each model combination
projection_data = {}

for model_key, files in model_groups.items():
    try:
        # Combine all time chunks for this model combination
        # Use combine='nested' with concat_dim to properly concatenate along time dimension
        ds = xr.open_mfdataset(sorted(files), combine='nested', concat_dim='time')
        # Extract EL64 data
        ds_el64 = ds.sel(nuts=admin_id)
        projection_data[model_key] = ds_el64
    except Exception as e:
        print(f"Error loading {model_key}: {e}")

print(f"\nSuccessfully loaded {len(projection_data)} model combinations")

# Show example
if len(projection_data) > 0:
    example_key = list(projection_data.keys())[0]
    print(f"\nExample dataset ({example_key}):")
    print(projection_data[example_key])


Successfully loaded 17 model combinations

Example dataset (earth_hirham5_rcp4_5):
<xarray.Dataset> Size: 10kB
Dimensions:  (time: 600)
Coordinates:
  * time     (time) datetime64[ns] 5kB 1951-01-01 1952-01-01 ... 2100-01-01
    nuts     <U4 16B 'EL64'
Data variables:
    dmd      (time) float64 5kB dask.array<chunksize=(150,), meta=np.ndarray>


## Save Regional Hazard Data

Finally, we'll save all the timeseries data to CSV files for further analysis.

In [ ]:
# Save historical data to CSV
if 'drought_dur_hist_el64' in locals():
    hist_df = drought_dur_hist_el64['dmd'].to_dataframe().reset_index()
    hist_df['scenario'] = 'historical'
    hist_df['model'] = 'reanalysis'
    hist_csv = output_dir / f"drought_duration_historical_{admin_id}.csv"
    hist_df.to_csv(hist_csv, index=False)
    print(f"Saved historical data to: {hist_csv}")
    print(f"  Shape: {hist_df.shape}")
    print(f"  Columns: {list(hist_df.columns)}")

Saved historical data to: data/EL64/drought_hazard/drought_duration_historical_EL64.csv
  Shape: (336, 6)
  Columns: ['time', 'realization', 'nuts', 'dmd', 'scenario', 'model']


In [ ]:
# Save projection data to CSV
if len(projection_data) > 0:
    all_proj_data = []
    
    for model_key, ds in projection_data.items():
        # Parse model information
        parts = model_key.split('_')
        gcm = parts[0]
        rcm = parts[1]
        rcp = parts[2] + '_' + parts[3]  # e.g., rcp4_5
        ens = '_'.join(parts[4:])  # e.g., r1i1p1
        
        # Convert to dataframe
        df = ds['dmd'].to_dataframe().reset_index()
        df['scenario'] = rcp
        df['gcm'] = gcm
        df['rcm'] = rcm
        df['ensemble'] = ens
        df['model'] = model_key
        
        all_proj_data.append(df)
    
    # Combine all projection data
    proj_df = pd.concat(all_proj_data, ignore_index=True)
    
    # Save to CSV
    proj_csv = output_dir / f"drought_duration_projections_{admin_id}.csv"
    proj_df.to_csv(proj_csv, index=False)
    print(f"\nSaved projection data to: {proj_csv}")
    print(f"  Shape: {proj_df.shape}")
    print(f"  Columns: {list(proj_df.columns)}")
    print(f"  Unique models: {proj_df['model'].nunique()}")
    print(f"  Scenarios: {proj_df['scenario'].unique()}")


Saved projection data to: data/EL64/drought_hazard/drought_duration_projections_EL64.csv
  Shape: (10340, 8)
  Columns: ['time', 'nuts', 'dmd', 'scenario', 'gcm', 'rcm', 'ensemble', 'model']
  Unique models: 17
  Scenarios: ['rcp4_5' 'rcp8_5' 'wrf381p_rcp4' 'wrf381p_rcp8' 'cclm4_8' 'rca4_rcp4'
 'rca4_rcp8']


In [ ]:
# Create a combined CSV with both historical and projections
if 'hist_df' in locals() and 'proj_df' in locals():
    # Align columns
    common_cols = ['time', 'nuts', 'dmd', 'scenario', 'model']
    
    hist_df_clean = hist_df[common_cols].copy()
    proj_df_clean = proj_df[['time', 'nuts', 'dmd', 'scenario', 'model']].copy()
    
    combined_df = pd.concat([hist_df_clean, proj_df_clean], ignore_index=True)
    combined_df = combined_df.sort_values('time').reset_index(drop=True)
    
    combined_csv = output_dir / f"drought_duration_all_{admin_id}.csv"
    combined_df.to_csv(combined_csv, index=False)
    print(f"\nSaved combined data to: {combined_csv}")
    print(f"  Shape: {combined_df.shape}")
    print(f"  Time range: {combined_df['time'].min()} to {combined_df['time'].max()}")


Saved combined data to: data/EL64/drought_hazard/drought_duration_all_EL64.csv
  Shape: (10676, 5)
  Time range: 1940-01-01 00:00:00 to 2100-01-01 00:00:00


### Output summary 

**Output files created:**
- `drought_duration_historical_{admin_id}.csv` - Historical drought duration
- `drought_duration_projections_{admin_id}.csv` - All projections of drought duration
- `drought_duration_all_{admin_id}.csv` - Combined historical and projections

These CSV files can now be used for further analysis, risk assessment, or integration with exposure and vulnerability data.